In [2]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import os
import json
import torch 
import torch.nn as nn

pd.set_option("display.max_colwidth", None)



In [3]:
#Get the videos json
videos = os.path.join("..", "Videos.json")

#Open the json
with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

#Turn the json into a pandas dataframe with proper headings     
df = pd.json_normalize(data["videos"])

print(df.columns)
print(df.head())


Index(['id', 'Sign', 'filepath', 'HamNoSys', 'concept_url', 'HandLandmark'], dtype='str')
   id      Sign  \
0   1     April   
1   2    Athens   
2   3    August   
3   4    Berlin   
4   5  February   

                                                             filepath  \
0     C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
1    C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\Athens.mp4   
2    C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\August.mp4   
3    C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\Berlin.mp4   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\February.mp4   

                                          HamNoSys  \
0     
1                 
2         
3                                         
4                             

                               

In [4]:
#find the path to each landmark json in the videos json
Hand_Landmarks_paths = df["HandLandmark"]

Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]


In [5]:

#Empty list for storing each dataframe of the handlandmarks
Hand_landmarks_dfs = []

for path in Hand_Landmarks:
    
    with open(path, "r") as f:
        data = json.load(f)
        
    #Create dataframes
    temp_df = pd.json_normalize(data)
    temp_df["source"] = path
    
    Hand_landmarks_dfs.append(temp_df)

In [6]:
#Take each handlandmark df and make it into one big df
Hand_Landmark_df = pd.concat(Hand_landmarks_dfs, ignore_index=True)

In [8]:
#Hand_Landmark_df.head(1)



In [10]:
rows = []

#For each video
for _, video_row in Hand_Landmark_df.iterrows():
    video_path = video_row["video_path"]#Get each video path
    fps = video_row["fps"]#Get the fps for those videos
    source = video_row.get("source",None)#
    
    #for all the frames in current video
    for frame in video_row["frames"]:
        frame_index = frame["frame_index"] #Set frame index
        time_sec = frame["time_sec"]#Set the time of frame
        
        #Then for each hand in current frame
        for hand in frame["hands"]:
            handedness = hand["handedness"]#which hand
            handedness_score = hand["handedness_score"]#How sure that this is the right hand
            hand_index = hand["hand_index"]#The index of hand
            
            #For each landmark in hand
            for lm in hand["landmarks"]:
                rows.append({
                    "video_path": video_path,
                    "fps": fps,
                    "source": source,
                    "frame_index": frame_index,
                    "time_sec": time_sec,
                    "hand_index": hand_index,
                    "handedness": handedness,
                    "handedness_score": handedness_score,
                    "landmark_id": lm["id"],
                    "landmark_name": lm["name"],
                    "x": lm["x"],
                    "y": lm["y"],
                    "z": lm["z"],
                })
                
Hand_landmarks_df = pd.DataFrame(rows)

In [11]:
print(Hand_landmarks_df.head())

                                                        video_path   fps  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   

                                                                  source  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   

   frame_index  time_sec  hand_index handedness  handedness_score  \
0            0   

In [12]:
#Need to piivot table so that it is done by frame instead of landmark

wide_df = Hand_landmarks_df.pivot_table(
    index=["video_path", "frame_index", "time_sec", "source"],
    columns=["handedness", "landmark_name"],
    values=["x", "y", "z"]
)

#Create readable headings
wide_df.columns = [
    f"{hand}_{landmark}_{axis}"
    for axis, hand, landmark in wide_df.columns
]

wide_df = wide_df.reset_index()
wide_df = wide_df.fillna(0.0)

In [13]:
print(wide_df.head())
print(wide_df.shape)

                                                        video_path  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   

   frame_index  time_sec  \
0            0      0.00   
1            1      0.04   
2            5      0.20   
3            6      0.24   
4            7      0.28   

                                                                  source  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
4  C:\Users\mccor\Documents\Y4

In [14]:
feature_cols = [col for col in wide_df.columns
                if col not in ["video_path", "frame_index", "time_sec","source"]]

sequences = []
video_ids = []

for video, group in wide_df.groupby("video_path"):
    
    group = group.sort_values("frame_index")
    seq = group[feature_cols].values
    sequences.append(seq)
    video_ids.append(video)
    
print(len(sequences)) 
import numpy as np

feature_cols = [col for col in wide_df.columns 
                if col not in ["video_path", "frame_index", "time_sec", "source"]]

sequences = []
video_ids = []

for video, group in wide_df.groupby("video_path"):
    
    # sort frames in correct order
    group = group.sort_values("frame_index")
    
    # extract features only
    seq = group[feature_cols].values   # shape: [num_frames, 126]
    
    sequences.append(seq)
    video_ids.append(video)

print(len(sequences))
print(sequences[0].shape)

    

1008
1008
(239, 126)


In [15]:
max_len = max(seq.shape[0] for seq in sequences)
num_features = sequences[0].shape[1]

x = np.zeros((len(sequences), max_len, num_features))

for i, seq in enumerate(sequences):
    length = seq.shape[0]
    x[i, :length, :] = seq

print(x.shape)
    
    

(1008, 1951, 126)


In [16]:
X_tensor = torch.tensor(x, dtype=torch.float32)

print(X_tensor.shape)

torch.Size([1008, 1951, 126])


In [ ]:
labels = []

for vid in video_ids:
    filename = os.path.basename(vid)
 

['April', 'Athens', 'August', 'Berlin', 'February']


In [27]:
#

In [28]:
#


In [12]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam #Do I keep adam or GD

import lightning as L
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
y_tensor = torch.tensor(y, dtype=torch.long)


torch.Size([1008])


In [26]:
print(X_tensor.shape)
print(y_tensor.shape)
print(len(label_to_id))

torch.Size([1008, 1951, 126])
torch.Size([1008])
1008


In [ ]:

class LSTM(L.LightningModel):
   def __init__(self):
       super().init__()
       
       mean = torch.tensor(0.0)
       std = torch.tensor(1.0)
       
       #How to create a equal number of weights to input videos frames 
    
   def lstm_unit(self, input_value, long_memory,short_memory):
       pass
   
   def forward(self, input):
       pass
   
   def configure_optimisers(self):
       pass
   
   def training_step(self,batch,batch_idx):
       pass